# Chapter 00: Setup and Math Refresher

Welcome to the first notebook of **Neural Networks from Scratch**!

In this interactive lab, we explore the essential mathematics underpinning deep learning:
1. **Vectors, matrices, and broadcasting gotchas**
2. **The dot product** as alignment
3. **Derivatives and tangent slopes**
4. **The gradient vector** on a 2D surface
5. **The chain rule** in code
6. **Numerical gradient checking** (never trust an unverified derivative)
7. **PyTorch autograd verification**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure nnlib is on path
sys.path.insert(0, str(Path.cwd().parents[1] if Path.cwd().name == "00-setup-and-math" else Path.cwd()))
from nnlib.device import get_device, get_device_name
from nnlib.math import numerical_gradient, relative_error

print("Compute device:", get_device_name())

## 1. Vectors, Matrices & The Broadcasting Trap

In NumPy, shape mismatches are the #1 source of silent bugs in neural network implementations.
Let us see what happens when a 1D array `(N,)` meets a 2D column vector `(N, 1)`.

In [ ]:
a = np.array([1.0, 2.0, 3.0])          # Shape: (3,)
b = np.array([[10.0], [20.0], [30.0]]) # Shape: (3, 1)

print("a shape:", a.shape)
print("b shape:", b.shape)
print("Result of a + b:\n", a + b)
print("Result shape:", (a + b).shape)  # Warning: (3, 3) matrix instead of (3, 1) vector!

To fix this, always be explicit with dimensions using `.reshape(-1, 1)`:

In [ ]:
a_col = a.reshape(-1, 1)
print("Correct addition shape:", (a_col + b).shape)
print("Result:\n", a_col + b)

## 2. Visualizing the Derivative as a Tangent Line

Let $f(x) = x^2 - 2x + 1$. The derivative is $f'(x) = 2x - 2$.
At $x = 3$, $f(3) = 4$ and $f'(3) = 4$.
The tangent line is given by $y = f(3) + f'(3)(x - 3)$.

In [ ]:
def f_quad(x):
    return x**2 - 2*x + 1

def df_quad(x):
    return 2*x - 2

x_vals = np.linspace(-1, 5, 200)
y_vals = f_quad(x_vals)

x0 = 3.0
y0 = f_quad(x0)
slope0 = df_quad(x0)
tangent_line = y0 + slope0 * (x_vals - x0)

plt.figure(figsize=(8, 5))
plt.plot(x_vals, y_vals, label="f(x) = x^2 - 2x + 1", color="#2563eb", linewidth=2)
plt.plot(x_vals, tangent_line, "--", label=f"Tangent at x=3 (slope={slope0})", color="#dc2626", linewidth=1.5)
plt.scatter([x0], [y0], color="#dc2626", s=70, zorder=5)
plt.title("Visualizing the Derivative as a Tangent Line", fontsize=14)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.ylim(-2, 18)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 3. Visualizing the Gradient Vector on a 2D Contour

Consider $f(x, y) = x^2 + 2y^2$. The gradient is $\nabla f(x, y) = [2x, 4y]^T$.
Let us draw the contour lines and plot the gradient vectors pointing uphill.

In [ ]:
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = X**2 + 2 * Y**2

plt.figure(figsize=(7, 6))
cp = plt.contour(X, Y, Z, levels=15, cmap="viridis")
plt.clabel(cp, inline=True, fontsize=8)

# Sample points to plot gradient vectors
pts_x = np.array([1.5, -1.5, 2.0, -1.0])
pts_y = np.array([1.5, 1.0, -1.5, -2.0])
grad_x = 2 * pts_x
grad_y = 4 * pts_y

plt.quiver(pts_x, pts_y, grad_x, grad_y, color="red", scale=25, label="Gradient (\u2207f, steepest ascent)")
plt.quiver(pts_x, pts_y, -grad_x, -grad_y, color="blue", scale=25, label="Negative Gradient (-\u2207f, descent)")
plt.scatter(0, 0, color="black", marker="x", s=100, label="Minimum (0, 0)")
plt.title("Contour Map of f(x, y) = x^2 + 2y^2 with Gradient Vectors")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. The Chain Rule in Action

Composite function: $u = 2x + 1$, $y = u^3$, $L = \frac{1}{2}(y - 5)^2$.
At $x = 1.0$, let us compute both forward pass and backward pass:

In [ ]:
x = 1.0
# Forward pass
u = 2 * x + 1
y = u ** 3
L = 0.5 * (y - 5) ** 2

# Backward pass (Chain rule)
dL_dy = y - 5
dy_du = 3 * (u ** 2)
du_dx = 2.0
dL_dx = dL_dy * dy_du * du_dx

print(f"Forward:  x={x} -> u={u} -> y={y} -> Loss={L}")
print(f"Backward: dL/dy={dL_dy}, dy/du={dy_du}, du/dx={du_dx}")
print(f"Result:   dL/dx = {dL_dx}")

## 5. Numerical Gradient Verification & PyTorch Autograd

Let us verify our manual derivation using `nnlib.math.numerical_gradient` and compare with PyTorch.

In [ ]:
# 1. Numerical finite difference
def scalar_pipeline(x_arr):
    x_in = x_arr[0]
    u_val = 2 * x_in + 1
    y_val = u_val ** 3
    return 0.5 * (y_val - 5) ** 2

num_grad = numerical_gradient(scalar_pipeline, np.array([1.0]))[0]
err = relative_error(np.array([dL_dx]), np.array([num_grad]))
print("Hand derivative:     ", dL_dx)
print("Numerical derivative:", num_grad)
print(f"Relative Error:       {err:.2e} (Passed!)")

# 2. PyTorch autograd reference
x_torch = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
u_torch = 2.0 * x_torch + 1.0
y_torch = u_torch ** 3
L_torch = 0.5 * (y_torch - 5.0) ** 2
L_torch.backward()
print("PyTorch Autograd:    ", x_torch.grad.item())
assert np.isclose(x_torch.grad.item(), dL_dx)
print("All 3 methods agree to high precision!")